In [2]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 17.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [3]:
from datasets import load_dataset

ds = load_dataset("microsoft/ms_marco", "v1.1")
test = ds["test"]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/9.48k [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/175M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10047 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/82326 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9650 [00:00<?, ? examples/s]

In [7]:
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

In [10]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


# BOW

In [11]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    words = word_tokenize(text)
    words = [word for word in words if word not in stop_words]
    words = [stemmer.stem(word) for word in words]
    return ' '.join(words)

corpus = []
queries = []
labels = []
query_ids = []

for sample in test:
    query_id = sample['query_id']
    query = normalize_text(sample['query'])
    passages = [normalize_text(p) for p in sample['passages']['passage_text']]
    is_selected = sample['passages']['is_selected']

    queries.append(query)
    corpus.append(query)
    corpus.extend(passages)

    labels.append(is_selected)
    query_ids.append(query_id)

In [ ]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)

results = []
for idx, query in enumerate(queries):
    query_vector = vectorizer.transform([query])
    passage_vectors = X[len(queries):]

    similarities = cosine_similarity(query_vector, passage_vectors).flatten()

    ranked_passages = sorted(
        enumerate(similarities),
        key=lambda x: x[1],
        reverse=True
    )

    results.append({
        "query_id": query_ids[idx],
        "ranked_passages": ranked_passages
    })

In [ ]:
for result, label in zip(results, labels):
    relevant_indices = [i for i, is_sel in enumerate(label) if is_sel]
    predicted_indices = [idx for idx, _ in result["ranked_passages"]]

    print(f"Query ID: {result['query_id']}")
    print(f"Relevant passages: {relevant_indices}")
    print(f"Predicted order: {predicted_indices[:5]}")
    print("-" * 50)

# BERT

In [ ]:
import torch
from transformers import BertTokenizer, BertModel

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def get_bert_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    inputs = {key: value.to(device) for key, value in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].cpu().numpy()

corpus_embeddings = [get_bert_embedding(text) for text in corpus]
corpus_embeddings = torch.vstack(corpus_embeddings).numpy()

In [ ]:
results = []

for idx, query in enumerate(queries):
    query_embedding = get_bert_embedding(query)
    passage_embeddings = corpus_embeddings[len(queries):]

    similarities = cosine_similarity(query_embedding, passage_embeddings).flatten()

    ranked_passages = sorted(
        enumerate(similarities),
        key=lambda x: x[1],
        reverse=True
    )

    results.append({
        "query_id": query_ids[idx],
        "ranked_passages": ranked_passages
    })

for result, label in zip(results, labels):
    relevant_indices = [i for i, is_sel in enumerate(label) if is_sel]
    predicted_indices = [idx for idx, _ in result["ranked_passages"]]

    print(f"Query ID: {result['query_id']}")
    print(f"Relevant passages: {relevant_indices}")
    print(f"Predicted order: {predicted_indices[:5]}")
    print("-" * 50)